# LIKAS — Fine-Tuning Gemma 4 E2B for an Offline Disaster Assistant

>  Your companion when calamity strikes the nation

***Likas, in Filipino, means nature — yet it also means to evacuate. An AI-powered platform that helps citizens evacuate during disasters caused by nature.***

Notebook of John Paul M. Curada from Team Iskolars of Polytechnic University of the Philippines

___

**LIKAS** is an offline disaster companion for the Philippines. It fine-tunes Google's `gemma-4-e2b-it` so every reply is one clean JSON object — either a **tool call** (look up a protocol, find a shelter, route to evacuation) or a **message** to the user. That strict format is what lets a 5B model run safely on a phone with no network.

### Why it matters

The Philippines faces ~20 typhoons a year plus earthquakes and active volcanoes. When disaster strikes, cell towers and power fail *exactly* when people need guidance most. LIKAS turns Gemma 4 into a **specialist**, not a chatbot: it quotes authoritative **NDRRMC / PHIVOLCS / PAGASA** protocols, routes to the nearest evacuation center, and answers in **Filipino, English, or Taglish** — fully offline.

### How it's built

- **`Unsloth`** — fine-tuning a 5B model on one free Kaggle T4 is only practical with Unsloth: 4-bit QLoRA (~1.2% of weights trained), ~2× faster training, and memory-efficient gradient checkpointing. The full fine-tune finishes in under an hour on a single GPU.
- **`llama.cpp`** — the assistant must run *on the user's phone* with no server, GPU, or internet. llama.cpp converts and quantizes the trained model to a 4-bit `GGUF` (~1.8 GB) that runs on-device via **`llama.rn`** (its React Native binding). *Unsloth trains it; llama.cpp ships it to the phone.*

### At a glance

| | |
|---|---|
| **Base model** | `google/gemma-4-e2b-it` |
| **Fine-tuning** | `Unsloth` — 4-bit QLoRA, ~1.2% params trained, ~2× faster on one T4 |
| **On-device runtime** | `llama.cpp` → `GGUF` Q4_K_M, served through `llama.rn` |
| **Output** | `likas-q4_k_m.gguf` (~1.8 GB) — fits and runs on a phone, offline |
| **Hardware** | trained on Kaggle T4 GPU; runs inference on phone CPU |
| **Tracking** | train + validation loss streamed to Weights & Biases |

### Pipeline

`setup → load model → load data → format → Unsloth train → push LoRA to Hub → build GGUF (clean session) → smoke test`

> ⚠️ **Before running:** add `HF_TOKEN` and `WANDB_API_KEY` under **Add-ons → Secrets** (Section I).
>
> 💾 **Disk note:** Kaggle gives ~20 GB in `/kaggle/working`, but a 5B model's fp16 merge needs most of that — too much *after* training fills the disk. So this notebook **trains and pushes the LoRA adapter here**, then builds the GGUF in a **fresh session** with a clean disk (Section X). The LoRA *is* the fine-tune; the GGUF is reproducible from it.

## I. Setup

Three cells run in order: **install dependencies**, **import everything**, **define all constants**. Every path and tunable lives in the constants cell — change settings there, not further down.

> **Before running:** add your secrets in the Kaggle right-hand panel → **Add-ons → Secrets**. Required:
> - `HF_TOKEN` — Hugging Face write token (for pushing the model)
> - `WANDB_API_KEY` — Weights & Biases key (for loss tracking)
>
> The constants cell reads these directly. If a secret is missing, that cell will error — add it and re-run.

In [ ]:
%%capture
# Install dependencies (Kaggle environment). Version pins matter:
# transformers 5.5 needs huggingface_hub >= 1.5, and Unsloth's trl pin
# needs datasets >= 4.7.
import re, torch

v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")

!pip install -q sentencepiece protobuf "datasets>=4.7.0" "huggingface_hub>=1.5.0,<2.0" hf_transfer \
    tyro msgspec cut_cross_entropy
!pip install -q "trl>=0.18.2,<=0.24.0,!=0.19.0"
!pip install -q --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft triton unsloth
!pip install -q --no-deps --upgrade "torchao>=0.16.0"
!pip install -q --no-deps transformers==5.5.0 "tokenizers>=0.22.0,<=0.23.0"
!pip install -q -U "huggingface_hub>=1.5.0,<2.0"
!pip install -q wandb

In [ ]:
# All imports for the whole notebook. Unsloth must be imported before
# transformers so its speed patches apply.
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

import os
import glob
import json
import shutil
import urllib.request

import torch
import wandb
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig
from transformers import TextStreamer
from huggingface_hub import HfApi, login, create_repo

# Allow many recompiles before falling back to eager mode (sequence lengths vary).
torch._dynamo.config.recompile_limit = 64

print("Imports OK. Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())

In [ ]:
# --- Constants: all tunables and paths live here ---
from kaggle_secrets import UserSecretsClient

_secrets = UserSecretsClient()
os.environ["WANDB_API_KEY"] = _secrets.get_secret("WANDB_API_KEY")
HF_TOKEN = _secrets.get_secret("HF_TOKEN")

# Paths
WORKDIR = "/kaggle/working"
DATA_DIR = "/kaggle/input/datasets/jeypiic/likas-ai-datasets"   # train.jsonl + test.jsonl
MODEL_CARD_PATH = f"{DATA_DIR}/MODEL_CARD.md"
MODEL_NAME = "/kaggle/input/models/google/gemma-4/transformers/gemma-4-e2b-it/1"
OUTPUT_DIR = f"{WORKDIR}/outputs"
LORA_DIR = f"{WORKDIR}/likas_gemma4_e2b_lora"
MERGED_DIR = f"{WORKDIR}/likas_merged_fp16"
GGUF_PATH = f"{WORKDIR}/likas-q4_k_m.gguf"

# Model / training
MAX_SEQ_LENGTH = 4096
LORA_R = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0
LEARNING_RATE = 5e-5
NUM_EPOCHS = 2
SEED = 3407

# Weights & Biases
WANDB_PROJECT = "likas-gemma4-e2b"
WANDB_RUN_NAME = "v4-toolcalling-sft"
os.environ["WANDB_PROJECT"] = WANDB_PROJECT
os.environ["WANDB_LOG_MODEL"] = "false"   # don't upload checkpoints (saves Kaggle disk)
os.environ["WANDB_WATCH"] = "false"

# Hugging Face target repos
HF_USER = "jpcurada"
LORA_REPO = f"{HF_USER}/likas-gemma4-e2b-lora"
FP16_REPO = f"{HF_USER}/likas-gemma4-e2b-fp16"
GGUF_REPO = f"{HF_USER}/likas-gemma4-e2b-gguf"

print("Constants set. Data:", DATA_DIR)
print("Outputs ->", WORKDIR)

## II. Log in to Weights & Biases

`WANDB_API_KEY` was already loaded from Kaggle Secrets in the constants cell. This just logs in so training metrics stream to your dashboard.

In [ ]:
wandb.login(key=os.environ["WANDB_API_KEY"])
print("Logged in to Weights & Biases.")

## III. Load the base model with Unsloth

`FastLanguageModel.from_pretrained` is Unsloth's optimized loader — it patches Gemma 4's attention and MLP kernels for ~2× faster training and lower VRAM, and loads the weights in **4-bit** so a 5B model fits on a 16 GB T4.

`get_peft_model` then attaches **LoRA** adapters: instead of updating all 5B weights, we train small low-rank matrices on the attention + MLP projections — about **1.2% of parameters**. That is what makes a full fine-tune feasible on free hardware and keeps the adapter tiny (~110 MB). `use_gradient_checkpointing="unsloth"` trades a little compute for a lot of memory so the 4096-token context fits.

In [ ]:
print(f"Loading model from: {MODEL_NAME}")

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
    full_finetuning = False,
    use_gradient_checkpointing = "unsloth",
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
    use_rslora = False,
    loftq_config = None,
)

## IV. Load the dataset

Reads `train.jsonl` and `test.jsonl` from the mounted Kaggle dataset. Each line is one conversation: `{"messages": [{role, content}, ...]}` with roles `system`, `user`, `assistant`, and `tool`.

In [ ]:
for f in ("train.jsonl", "test.jsonl"):
    assert os.path.exists(f"{DATA_DIR}/{f}"), f"No {f} in {DATA_DIR}"

ds = load_dataset("json", data_files={
    "train": f"{DATA_DIR}/train.jsonl",
    "test":  f"{DATA_DIR}/test.jsonl",
})
raw_train = ds["train"]
raw_test  = ds["test"]

print(raw_train)
print(raw_test)
print("Example roles:", [m["role"] for m in raw_train[0]["messages"]])

### Preview a few examples

Print the actual content of a few training conversations so you can eyeball the data before training — check that tool calls match the question, the final reply stays grounded in the tool result, and the language matches the user.

In [ ]:
def show_conversation(example, max_len=400):
    for m in example["messages"]:
        content = m["content"]
        if len(content) > max_len:
            content = content[:max_len] + " …"
        print(f"[{m['role'].upper()}] {content}")
    print("-" * 80)

N_PREVIEW = 3
print(f"=== First {N_PREVIEW} training conversations ===\n")
for i in range(N_PREVIEW):
    show_conversation(raw_train[i])

print(f"\n=== {N_PREVIEW} random training conversations ===\n")
import random
for i in random.Random(SEED).sample(range(len(raw_train)), N_PREVIEW):
    show_conversation(raw_train[i])

## V. Format the conversations

Apply Gemma-4's chat template to turn each conversation into one training string. Gemma's template has no `tool` role, so we fold each tool result into the next turn as a `<tool_response>...</tool_response>` block — the model still sees the tool output, just in a role the template accepts.

In [ ]:
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

def normalize_messages(messages):
    """Fold each tool result into the following turn so Gemma-4's template accepts it."""
    out = []
    pending_tool = None
    for m in messages:
        role, content = m["role"], m["content"]
        if role == "tool":
            pending_tool = content
            continue
        if pending_tool is not None and role in ("user", "assistant"):
            wrapped = f"<tool_response>\n{pending_tool}\n</tool_response>"
            if role == "user":
                content = wrapped + "\n\n" + content
            else:
                out.append({"role": "user", "content": wrapped})
            pending_tool = None
        out.append({"role": role, "content": content})
    if pending_tool is not None:
        out.append({"role": "user", "content": f"<tool_response>\n{pending_tool}\n</tool_response>"})
    return out

def format_row(example):
    msgs = normalize_messages(example["messages"])
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    return {"text": text}

train_ds = raw_train.map(format_row, remove_columns=raw_train.column_names)
test_ds  = raw_test.map(format_row,  remove_columns=raw_test.column_names)

print(train_ds[0]["text"][:1200])

## VI. Train (Unsloth-accelerated SFT)

Supervised fine-tuning through `trl`'s `SFTTrainer` running on the **Unsloth-patched** model from Section III — the patched kernels are why ~150 steps over 2 epochs finish in well under an hour on one T4. Loss is computed over the full conversation. Runs the full 2 epochs (no early stopping).

- **Loss tracking:** train and validation loss are logged to W&B every 10 steps.
- **Best model:** `load_best_model_at_end` reloads the checkpoint with the lowest validation loss, so the saved model is the best one — not the last (possibly overfit) step.
- **Tuning:** to fight overfitting, lower `LEARNING_RATE` or raise `LORA_DROPOUT` in the constants cell, then re-run from here.

In [ ]:
wandb.init(
    project = WANDB_PROJECT,
    name = WANDB_RUN_NAME,
    config = {
        "model": MODEL_NAME,
        "dataset": DATA_DIR,
        "max_seq_length": MAX_SEQ_LENGTH,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
        "learning_rate": LEARNING_RATE,
        "epochs": NUM_EPOCHS,
    },
)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = test_ds,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_ratio = 0.03,
        num_train_epochs = NUM_EPOCHS,
        learning_rate = LEARNING_RATE,

        # Log + eval + save every 10 steps. save_steps must equal eval_steps
        # because load_best_model_at_end=True requires aligned cadences.
        logging_steps = 10,
        logging_first_step = True,
        eval_strategy = "steps",
        eval_steps = 10,
        save_strategy = "steps",
        save_steps = 10,
        per_device_eval_batch_size = 2,
        # Only 1 checkpoint on disk at a time (~3-4GB). The Kaggle
        # /kaggle/working quota is ~20GB and the merged fp16 is ~10GB, so more
        # than one checkpoint would overflow. load_best_model_at_end still
        # works with limit=1 — it always retains the single best checkpoint,
        # which Section VII deletes right after loading it back into memory.
        save_total_limit = 1,
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
        greater_is_better = False,

        optim = "adamw_8bit",
        weight_decay = 0.001,
        lr_scheduler_type = "cosine",
        max_grad_norm = 0.3,
        seed = SEED,
        output_dir = OUTPUT_DIR,
        report_to = "wandb",
        run_name = WANDB_RUN_NAME,
        max_length = MAX_SEQ_LENGTH,
    ),
)

In [ ]:
gpu = torch.cuda.get_device_properties(0)
start_mem = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
print(f"GPU = {gpu.name}. Max memory = {round(gpu.total_memory/1024**3, 3)} GB.")
print(f"{start_mem} GB reserved before training.")

In [ ]:
trainer_stats = trainer.train()

In [ ]:
used_mem = round(torch.cuda.max_memory_reserved() / 1024**3, 3)
print(f"Train runtime: {trainer_stats.metrics['train_runtime']:.1f}s")
print(f"Peak reserved memory: {used_mem} GB")

wandb.finish()  # close the run in the W&B UI

## VII. Save the LoRA adapter

Free the trainer checkpoints, then save **only the LoRA adapter** (~110 MB). We deliberately do **not** save a merged fp16 model here — a 9.5 GB fp16 copy plus split overhead does not fit in Kaggle's ~20 GB `/kaggle/working` quota (it crashed the last two runs). The GGUF step in Section VIII merges + quantizes in one streaming pass instead, so fp16 never touches disk.

In [ ]:
def disk_free_gb(path="/kaggle/working"):
    total, used, free = shutil.disk_usage(path)
    return round(free / 1024**3, 2)

print(f"Free disk before cleanup: {disk_free_gb()} GB")

# Best model is already in memory (load_best_model_at_end). The trainer
# checkpoints are dead weight now — delete to reclaim disk.
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
    print(f"Deleted {OUTPUT_DIR}")
print(f"Free disk after cleanup:  {disk_free_gb()} GB")

# Save ONLY the LoRA adapter (~110 MB). No merged fp16 — see Section VIII.
model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)
print(f"LoRA saved to {LORA_DIR}")
print(f"Free disk now: {disk_free_gb()} GB")

## VIII. Convert to GGUF with llama.cpp — the on-device step

This is the step that makes Gemma 4 deployable on resource-constrained hardware. A fine-tuned 5B model in fp16 is ~10 GB and needs a GPU — useless on a phone during a disaster.

Unsloth's `save_pretrained_gguf` runs the **`llama.cpp`** toolchain end-to-end — merge LoRA → convert to GGUF → **quantize to 4-bit Q4_K_M** — in one pass, writing **only** the final ~1.8 GB file. It never materializes a separate ~10 GB fp16 copy on disk, which is what overflowed Kaggle's ~20 GB quota in earlier runs.

Q4_K_M shrinks the model ~5.5× (~10 GB → ~1.8 GB) with minimal quality loss — small enough to fit a mid-range phone's RAM and run on CPU with no network. The resulting GGUF is loaded on-device by **`llama.rn`** (llama.cpp's React Native binding) inside the LIKAS app — the exact same llama.cpp runtime, compiled for ARM/Android.

In [ ]:
def disk_free_gb(path="/kaggle/working"):
    total, used, free = shutil.disk_usage(path)
    return round(free / 1024**3, 2)

print(f"Free disk before GGUF: {disk_free_gb()} GB")

# Unsloth runs the llama.cpp toolchain (merge -> convert -> quantize) in one
# streaming pass and writes ONLY the quantized GGUF. quantization_method
# "q4_k_m" => the ~1.8 GB on-device artifact. This avoids ever holding a
# ~10 GB fp16 copy, which is what crashed the disk in earlier runs.
GGUF_DIR = f"{WORKDIR}/gguf_out"

model.save_pretrained_gguf(
    GGUF_DIR,
    tokenizer,
    quantization_method = "q4_k_m",
)

# Normalize the output filename to GGUF_PATH (Unsloth names it after the dir).
import glob
produced = glob.glob(f"{GGUF_DIR}/*.gguf") + glob.glob(f"{GGUF_DIR}/**/*.gguf")
assert produced, f"No .gguf produced in {GGUF_DIR}"
src = max(produced, key=os.path.getsize)   # the Q4_K_M file
if os.path.abspath(src) != os.path.abspath(GGUF_PATH):
    shutil.move(src, GGUF_PATH)
# Drop any leftover intermediates Unsloth may have left in the dir.
shutil.rmtree(GGUF_DIR, ignore_errors=True)

print(f"GGUF written to {GGUF_PATH}")
print(f"Free disk after GGUF: {disk_free_gb()} GB")
!ls -lh {WORKDIR}/*.gguf

## IX. Push the LoRA adapter to Hugging Face

Pushes **`jpcurada/likas-gemma4-e2b-lora`** — the LoRA adapter (~110–270 MB) plus tokenizer and `MODEL_CARD.md` as `README.md`.

The LoRA *is* the fine-tune — it's the irreplaceable artifact, and `push_to_hub` uploads only the adapter (no 9.5 GB merge), so it works inside Kaggle's disk quota. The Q4_K_M GGUF is **not** built here: merging a 5B model to fp16 needs ~20 GB free, which this session doesn't have after training. Build the GGUF in a fresh Kaggle session (or locally) from this pushed adapter — see Section X.

In [ ]:
login(token=HF_TOKEN)
api = HfApi()

create_repo(LORA_REPO, repo_type="model", exist_ok=True, token=HF_TOKEN)
print(f"Repo ready: https://huggingface.co/{LORA_REPO}")

# push_to_hub uploads ONLY the adapter (~110-270 MB) — no 9.5 GB merge,
# so it fits in Kaggle's disk quota. This is the irreplaceable artifact.
print(f"\nPushing LoRA to {LORA_REPO} ...")
model.push_to_hub(LORA_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(LORA_REPO, token=HF_TOKEN)

api.upload_file(
    path_or_fileobj=MODEL_CARD_PATH,
    path_in_repo="README.md",
    repo_id=LORA_REPO,
    token=HF_TOKEN,
)

print(f"\nDone. The fine-tune is permanently saved:")
print(f"  https://huggingface.co/{LORA_REPO}")

## X. Build the GGUF in a CLEAN session (run separately)

⚠️ **Do not run this in the same session as training.** Merging a 5B model to fp16 needs ~20 GB free disk; a post-training session only has ~11 GB and will crash with `No space left on device`.

**How to get the on-device GGUF:**
1. Open a **fresh Kaggle notebook** (empty `/kaggle/working` = full ~20 GB).
2. Add `jpcurada/likas-gemma4-e2b-lora` (pushed in Section IX) as a dataset/input.
3. Run only the cell below. It loads base + the LoRA, then `save_pretrained_gguf` does merge → convert → quantize to the ~1.8 GB Q4_K_M and pushes it.

With a clean 20 GB disk the 9.5 GB fp16 intermediate fits, so this same code that fails here succeeds there. The cell also does a quick `llama-cpp-python` smoke test so you confirm the shipped GGUF emits valid tool-calling JSON.

In [ ]:
# ============================================================================
# RUN THIS IN A FRESH KAGGLE SESSION — not after training.
# Requires: jpcurada/likas-gemma4-e2b-lora added as input, and the same
# install + imports + constants cells (Sections I) run first.
# ============================================================================
import os, glob, shutil

def disk_free_gb(p="/kaggle/working"):
    _, _, f = shutil.disk_usage(p); return round(f / 1024**3, 2)

print(f"Free disk at start: {disk_free_gb()} GB  (need ~20 — must be a clean session)")

# 1. Load base model, then attach the trained LoRA from the Hub.
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "jpcurada/likas-gemma4-e2b-lora",
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = True,
)

# 2. Merge + convert + quantize -> Q4_K_M GGUF (the on-device artifact).
GGUF_DIR = f"{WORKDIR}/gguf_out"
model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method="q4_k_m")

produced = glob.glob(f"{GGUF_DIR}/**/*.gguf", recursive=True)
assert produced, f"No .gguf produced in {GGUF_DIR}"
src = max(produced, key=os.path.getsize)
if os.path.abspath(src) != os.path.abspath(GGUF_PATH):
    shutil.move(src, GGUF_PATH)
shutil.rmtree(GGUF_DIR, ignore_errors=True)
print(f"GGUF written: {GGUF_PATH}  | free disk: {disk_free_gb()} GB")
!ls -lh {GGUF_PATH}

# 3. Push the GGUF to its own repo.
login(token=HF_TOKEN)
api = HfApi()
create_repo(GGUF_REPO, repo_type="model", exist_ok=True, token=HF_TOKEN)
api.upload_file(path_or_fileobj=GGUF_PATH, path_in_repo="likas-q4_k_m.gguf",
                repo_id=GGUF_REPO, token=HF_TOKEN)
api.upload_file(path_or_fileobj=MODEL_CARD_PATH, path_in_repo="README.md",
                repo_id=GGUF_REPO, token=HF_TOKEN)
print(f"GGUF pushed: https://huggingface.co/{GGUF_REPO}")

# 4. Smoke test the shipped GGUF through llama.cpp (same engine as llama.rn).
!pip install -q llama-cpp-python
from llama_cpp import Llama
llm = Llama(model_path=GGUF_PATH, n_ctx=MAX_SEQ_LENGTH, n_gpu_layers=-1, verbose=False)
out = llm.create_chat_completion(
    messages=[
        {"role": "system", "content": "You are LIKAS, an offline disaster assistant for the Philippines. Reply with exactly one JSON object: a tool call or a message."},
        {"role": "user",   "content": "May lindol! Ano gagawin ko ngayon?"},
    ],
    max_tokens=96, temperature=0.0, stop=["<end_of_turn>"],
)
print("\nSmoke test reply:")
print(out["choices"][0]["message"]["content"])